# 5. Retrieval-augmented answers with provenance

Index de-identified chunks, retrieve the ones relevant to a question, optionally
rerank them by shared medical concepts, and generate an answer that carries a
`SourceRef` back to each chunk it used.

Needs `faiss-cpu` (part of `pip install "openbtk[retrieval]"`).

> Every note, patient, number and identifier in this notebook is **fictitious**. It runs offline, downloads no model, and uses no real patient data.

> To run **offline**, this notebook uses two small stand-ins: a hashing "encoder"
> and an extractive "LLM". Both implement OpenBTK's real provider interfaces, so
> swapping in PubMedBERT (`openbtk.embeddings.presets`) and a real model changes
> two lines, not the pipeline.

In [1]:
import os

# Keep OpenBTK's routine debug lines out of this notebook's output.
os.environ.setdefault("OPENBTK_LOG_LEVEL", "warning")

'warning'

## Two stand-in providers

In [2]:
import re
import zlib

import numpy as np

from openbtk.core.base import BaseEmbeddingProvider, BaseLLMProvider
from openbtk.core.schemas import LLMResponse


class HashingEmbedding(BaseEmbeddingProvider):
    """Hashes words into 128 buckets. Not semantic - a stand-in for a real model."""

    sends_data_offsite = False
    dimension = 128

    def embed(self, texts):
        out = np.zeros((len(texts), self.dimension), dtype=np.float32)
        for i, text in enumerate(texts):
            for word in re.findall(r"[a-z0-9]+", text.lower()):
                out[i, zlib.crc32(word.encode()) % self.dimension] += 1.0
        norms = np.linalg.norm(out, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return out / norms


class ExtractiveLLM(BaseLLMProvider):
    """Answers with the first numbered source it was given. Not a language model."""

    sends_data_offsite = False

    def generate(self, prompt, **kwargs):
        return LLMResponse(text=prompt[:80])

    def stream(self, prompt, **kwargs):
        yield prompt[:80]

    def chat(self, messages, **kwargs):
        first = re.search(
            r"\[1\] (.+?)(?:\n\n|\nQuestion:)", messages[-1].content, re.S
        )
        return LLMResponse(
            text=f"According to source [1]: {first.group(1)}" if first else "No source."
        )


embedding = HashingEmbedding()
embedding.embed_one("hello").shape

(128,)

## Chunks to index

Three synthetic notes, de-identified and chunked like in notebook 4. Each chunk
gets a list of concept ids (`cuis`) - here from a tiny hand-made dictionary; in
practice a concept linker such as medspaCy or scispaCy produces them.

In [3]:
from openbtk.data.clinical_text.chunking import SectionAwareChunker
from openbtk.data.clinical_text.preprocessing import DeidPreprocessor, SectionSegmenter
from openbtk.data.clinical_text.schemas import ClinicalTextRecord

NOTES = {
    "note-1": "Assessment:\nType 2 diabetes mellitus, hemoglobin A1c 8.2 percent. Continue metformin.\n",
    "note-2": "Assessment:\nSugar-free diet counselling given at the clinic visit.\n",
    "note-3": "Assessment:\nHypertension, blood pressure controlled on lisinopril.\n",
}
# A toy concept dictionary (phrase -> concept id). C0011849 = diabetes mellitus,
# C0020538 = hypertensive disease. "a1c" and "blood sugar" are how a note and a
# question can name diabetes without using the word.
CONCEPTS = {
    "diabetes": "C0011849",
    "a1c": "C0011849",
    "blood sugar": "C0011849",
    "hypertension": "C0020538",
}


def cuis(text):
    return sorted({cui for word, cui in CONCEPTS.items() if word in text.lower()})


chunker = SectionAwareChunker(max_tokens=60)
deid = DeidPreprocessor()
segment = SectionSegmenter()

chunks = []
for record_id, text in NOTES.items():
    record = ClinicalTextRecord(record_id=record_id, source="synthetic", text=text)
    record = segment.process(deid.process(record))
    chunks.extend(chunker.chunk(record))

[(c.chunk_id, c.text.strip()[:50]) for c in chunks]

[('note-1:0', 'Type 2 diabetes mellitus, hemoglobin A1c 8.2 perce'),
 ('note-2:0', 'Sugar-free diet counselling given at the clinic vi'),
 ('note-3:0', 'Hypertension, blood pressure controlled on lisinop')]

## Index them

Vector stores hold vectors and metadata, not documents, so each chunk's text goes
in the metadata under `"text"` (the key `RAGPipeline` reads).

In [4]:
from openbtk.retrieval.faiss import FAISSVectorStore

store = FAISSVectorStore(dimension=embedding.dimension, metric="ip")
store.upsert(
    ids=[c.chunk_id for c in chunks],
    vectors=embedding.embed([c.text for c in chunks]),
    metadata=[
        {
            "text": c.text,
            "record_id": c.record_id,
            "chunk_id": c.chunk_id,
            "cuis": cuis(c.text),
        }
        for c in chunks
    ],
)

QUESTION = "How is the blood sugar controlled?"
hits = store.query(embedding.embed_one(QUESTION), top_k=3)
for h in hits:
    print(f"{h.score:.2f}  {h.id:10} {h.metadata['text'].strip()[:56]}")

0.50  note-3:0   Hypertension, blood pressure controlled on lisinopril.
0.27  note-2:0   Sugar-free diet counselling given at the clinic visit.
0.00  note-1:0   Type 2 diabetes mellitus, hemoglobin A1c 8.2 percent. Co


Plain vector search puts the **hypertension** note first (it shares "blood" and
"controlled" with the question) and the diabetes note - the one that actually
answers it - **last**, because the diabetes note never uses the word "sugar".

## Concept reranking

`ConceptOverlapReranker` re-orders candidates by how many medical concepts they
share with the *question*, using the concept ids we stored. The question's
"blood sugar" and the note's "A1c" name the same concept, so the diabetes note is
promoted to the top.

In [5]:
from openbtk.retrieval.reranker import ConceptOverlapReranker

reranker = ConceptOverlapReranker(extract_concepts=cuis)
reranked = reranker.rerank(QUESTION, hits, top_k=3)
for h in reranked:
    print(f"{h.id:10} concepts {h.metadata['cuis']}")

assert reranked[0].id == "note-1:0"  # the diabetes note is now first

note-1:0   concepts ['C0011849']
note-3:0   concepts ['C0020538']
note-2:0   concepts []


## The full pipeline, with provenance

`RAGPipeline` embeds the question, retrieves a wider pool, reranks it, builds a
numbered-source prompt and calls the model. The answer carries the exact chunks
it was grounded in.

In [6]:
from openbtk.pipelines import RAGPipeline

rag = RAGPipeline(
    embedding=embedding,
    vectorstore=store,
    llm=ExtractiveLLM(),
    reranker=reranker,
    top_k=2,
)
answer = rag.ask(QUESTION)
print(answer.text)
print()
for ref in answer.sources:
    print("source:", ref.record_id, "/", ref.chunk_id)

According to source [1]: Type 2 diabetes mellitus, hemoglobin A1c 8.2 percent. Continue metformin.

source: note-1 / note-1:0
source: note-3 / note-3:0


`answer.sources` is what lets a human - or a
[groundedness check](06_guardrails_and_terminology.ipynb) - verify a claim
against the precise chunk it came from.

## Limits

* The hashing encoder is lexical, so this notebook shows the *plumbing*, not
  retrieval quality. Measure yours with the metrics in
  [notebook 7](07_evaluation.ipynb).
* `ConceptOverlapReranker` is only as good as the concept extractor you give it.